# AI Job Agent — PHASE 5: Candidate Matching

**Goal:** take `filtered_jobs.parquet` from Phase 4 and compare every surviving job with the **single live candidate** field by field. The result is a deterministic, explainable `match_score` from 0 to 100 — no embeddings and no LLM calls are made in this notebook.

**Single-candidate scope.** This notebook implements **PATH B (live candidate)**, matching Phase 4. Batch evaluation remains a Phase 9 concern.

```text
filtered_jobs.parquet (Phase 4)      candidate_profile.json (Phase 2)
                 │                              │
                 └──── field-by-field match ────┘
                                  │
                 skill_vocabulary.parquet (Phase 1)
                         IDF-weighted skills
                                  │
                         matched_jobs.parquet
                     (→ Phase 6 re-ranking)
```

The scoring evidence is preserved as explicit matched/missing lists. Phase 6 can use the numeric scores for re-ranking, while Phase 7 can use the evidence to explain *why* a job fits and *what is missing*.

---
## 1. Setup

In [ ]:
import json
import math
import re
import shutil
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 50)

WORK = Path("work")
OUT = Path("outputs")
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
print("RUN_ID:", RUN_ID)

---
## 2. Load Phase 4 output (filtered jobs)

Upload `filtered_jobs.parquet` from Phase 4's `outputs/live_candidate/` folder (or the extracted `phase4_outputs.zip`). It already contains the full Phase 1 job columns plus Phase 3/4 ranks, so there is no need to upload `jobs_prepared.parquet` again.

In [ ]:
from google.colab import files as _colab_files

print("Upload Phase 4's filtered_jobs.parquet (PATH B / live candidate):")
uploaded_p4 = _colab_files.upload()

P4_DIR = WORK / "phase4"
P4_DIR.mkdir(exist_ok=True)
for name, content in uploaded_p4.items():
    (P4_DIR / name).write_bytes(content)

p4_file = P4_DIR / "filtered_jobs.parquet"
assert p4_file.exists(), (
    "filtered_jobs.parquet not found among the uploaded files — "
    "upload the live-candidate output from Phase 4."
)
print("\nSaved:", p4_file.name)

In [ ]:
filtered_df = pd.read_parquet(p4_file)

required_identity_cols = {"job_id", "rank", "similarity", "post_filter_rank"}
missing = required_identity_cols - set(filtered_df.columns)
assert not missing, f"filtered_jobs.parquet is missing expected columns: {missing}"
assert filtered_df["job_id"].is_unique, "filtered_jobs.parquet contains duplicate job_id values"

required_matching_cols = {
    "hard_skills_norm", "soft_skills_norm", "tools_technologies_norm",
    "languages_norm", "required_skills_norm", "preferred_skills_norm",
    "experience_level", "education_level", "qualifications",
}
missing_matching = required_matching_cols - set(filtered_df.columns)
assert not missing_matching, (
    f"Missing Phase 1 structured columns: {missing_matching}. "
    "Re-run Phase 3/4 using the current jobs_prepared.parquet output."
)

print(f"Filtered jobs ready for matching: {len(filtered_df):,}")
if len(filtered_df):
    print(f"Similarity range: {filtered_df['similarity'].min():.4f} – "
          f"{filtered_df['similarity'].max():.4f}")
filtered_df.head(3)

---
## 3. Load candidate profile (Phase 2)

Upload `candidate_profile.json` from Phase 2's `outputs/live_candidate/` folder. Matching uses the normalized skill fields (`*_norm`) and the same controlled enums created in Phases 1 and 2. Identity fields such as name, email and phone are intentionally excluded from scoring.

In [ ]:
print("Upload candidate_profile.json (from Phase 2):")
uploaded_profile = _colab_files.upload()

P2_DIR = WORK / "phase2"
P2_DIR.mkdir(exist_ok=True)
for name, content in uploaded_profile.items():
    (P2_DIR / name).write_bytes(content)

profile_path = P2_DIR / "candidate_profile.json"
assert profile_path.exists(), "candidate_profile.json not found among the uploaded files."

candidate_profile = json.loads(profile_path.read_text(encoding="utf-8"))
required_profile_fields = {
    "candidate_id", "hard_skills_norm", "soft_skills_norm",
    "tools_technologies_norm", "languages_norm",
    "experience_level", "education_level", "qualifications",
}
missing_profile = required_profile_fields - set(candidate_profile)
assert not missing_profile, f"candidate_profile.json is missing expected fields: {missing_profile}"

print(f"Loaded candidate: {candidate_profile.get('candidate_id')} "
      f"({candidate_profile.get('most_recent_title')})")
print(f"Normalized skills: {len(candidate_profile.get('all_skills_norm') or [])}")
print(f"Years of experience: {candidate_profile.get('years_experience_total')}")
print(f"Education: {candidate_profile.get('education_level')}")

---
## 4. Load Phase 1 skill vocabulary

Upload `skill_vocabulary.parquet` from Phase 1. Its `idf` column implements collection-frequency weighting: rare, discriminative skills contribute more than ubiquitous skills. Negative numerical edge cases are clipped to zero; if every required skill has zero/unknown IDF, the function falls back to ordinary set recall rather than adding an arbitrary constant.

In [ ]:
print("Upload skill_vocabulary.parquet (from Phase 1):")
uploaded_vocab = _colab_files.upload()

P1_DIR = WORK / "phase1"
P1_DIR.mkdir(exist_ok=True)
for name, content in uploaded_vocab.items():
    (P1_DIR / name).write_bytes(content)

vocab_path = P1_DIR / "skill_vocabulary.parquet"
assert vocab_path.exists(), "skill_vocabulary.parquet not found among the uploaded files."

vocab_df = pd.read_parquet(vocab_path)
required_vocab_cols = {"skill_canonical", "idf"}
missing_vocab = required_vocab_cols - set(vocab_df.columns)
assert not missing_vocab, f"skill_vocabulary.parquet is missing columns: {missing_vocab}"
assert vocab_df["skill_canonical"].is_unique, "skill vocabulary contains duplicate canonical skills"

IDF_BY_SKILL = dict(zip(vocab_df["skill_canonical"].astype(str), vocab_df["idf"].astype(float)))
print(f"Vocabulary loaded: {len(IDF_BY_SKILL):,} canonical skills")

---
## 5. Configuration — evidence-based scoring contract

Phase 5 deliberately contains **no hand-chosen business weights**. `match_score` is the equal-weight mean of five non-redundant evidence dimensions, which is the neutral parameter-free baseline when no labelled relevance data exists. `objective_rank_score` uses CRITIC objective weighting: each criterion's weight is derived from its contrast across the surviving jobs and its non-redundancy with the other criteria.

Education level remains an eligibility diagnostic because Phase 4 already enforces it; only education-field relevance can contribute here. Preferred skills are reported and used as a deterministic tie-breaker, not mixed into mandatory-skill coverage with an invented ratio. Phase 9 must eventually replace this baseline with weights learned from recruiter labels and evaluated with NDCG@K.

In [ ]:
RANKING_CRITERIA = [
    "core_skills_score",
    "languages_score",
    "experience_fit_score",
    "education_field_score",
    "qualifications_score",
]

CONFIG = {
    "weighting_method": "CRITIC objective weighting + equal-weight absolute score",
    "phase6_pool_size": 50,
}

print("Ranking criteria:", RANKING_CRITERIA)
print(json.dumps(CONFIG, indent=2))

---
## 6. Normalization and comparison helpers

Skills are already canonicalized by the identical Phase 1/2 normalization function, so Phase 5 compares them directly. Free-text fields such as certifications and education fields receive only lightweight deterministic normalization; semantic interpretation belongs in Phase 6.

In [ ]:
EXPERIENCE_HIERARCHY = ["Internship", "Entry Level", "Mid Level", "Senior Level", "Executive"]
EDUCATION_HIERARCHY = ["High School", "Diploma", "Bachelor's Degree", "Master's Degree", "PhD"]
_exp_order = {v.lower(): i for i, v in enumerate(EXPERIENCE_HIERARCHY)}
_edu_order = {v.lower(): i for i, v in enumerate(EDUCATION_HIERARCHY)}


def cell_values(value) -> list:
    """Convert a scalar/list/NumPy cell into a clean Python list."""
    if value is None:
        return []
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        items = list(value)
    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass
        items = [value]
    out = []
    for item in items:
        if item is None:
            continue
        try:
            if pd.isna(item):
                continue
        except (TypeError, ValueError):
            pass
        text = str(item).strip()
        if text and text.lower() not in {"nan", "none", "not specified"}:
            out.append(text)
    return out


def normalized_skill_set(value) -> set:
    # Inputs are *_norm fields; lower/strip is only defensive.
    return {str(v).strip().lower() for v in cell_values(value) if str(v).strip()}


def skill_weight(skill: str) -> float:
    """Non-negative inverse-document-frequency weight from Phase 1."""
    raw_idf = IDF_BY_SKILL.get(skill, 0.0)
    if not np.isfinite(raw_idf):
        return 0.0
    return max(float(raw_idf), 0.0)


def skill_coverage(job_skills, candidate_skills) -> dict:
    required = normalized_skill_set(job_skills)
    candidate = normalized_skill_set(candidate_skills)
    if not required:
        return {"score": None, "matched": [], "missing": [], "required_count": 0}

    matched = sorted(required & candidate)
    missing = sorted(required - candidate)
    weights = {skill: skill_weight(skill) for skill in required}
    total_weight = sum(weights.values())
    # Every job skill should exist in the Phase 1 vocabulary. If a legacy
    # file contains only unseen/zero-IDF skills, fall back to plain recall.
    if total_weight > 0:
        matched_weight = sum(weights[s] for s in matched)
        score = 100.0 * matched_weight / total_weight
    else:
        score = 100.0 * len(matched) / len(required)
    return {
        "score": round(score, 4),
        "matched": matched,
        "missing": missing,
        "required_count": len(required),
    }


def normalize_phrase(value: str) -> str:
    text = unicodedata.normalize("NFKC", str(value)).lower().strip()
    text = re.sub(r"[^\w+#.]+", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def phrase_similarity(left: str, right: str) -> float:
    a, b = normalize_phrase(left), normalize_phrase(right)
    if not a or not b:
        return 0.0
    a_tokens, b_tokens = set(a.split()), set(b.split())
    # Sørensen-Dice coefficient: bounded [0,1], symmetric, and contains
    # no manually selected partial-match constant.
    return 2.0 * len(a_tokens & b_tokens) / max(len(a_tokens) + len(b_tokens), 1)


def phrase_is_explicit_match(left: str, right: str) -> bool:
    a, b = normalize_phrase(left), normalize_phrase(right)
    if not a or not b:
        return False
    a_tokens, b_tokens = set(a.split()), set(b.split())
    # Exact equality or complete token containment is explainable evidence;
    # looser semantic relationships are deferred to Phase 6.
    return a_tokens == b_tokens or a_tokens.issubset(b_tokens) or b_tokens.issubset(a_tokens)


def is_education_credential(value: str) -> bool:
    """Degrees belong to Phase 4 education eligibility, not qualifications."""
    tokens = set(normalize_phrase(value).split())
    degree_markers = {"degree", "bachelor", "bachelors", "master", "masters",
                      "phd", "doctorate", "diploma", "school"}
    return bool(tokens & degree_markers)


def phrase_list_coverage(job_values, candidate_values) -> dict:
    requirements = cell_values(job_values)
    candidate_items = cell_values(candidate_values)
    if not requirements:
        return {"score": None, "matched": [], "missing": []}

    best_pairs = []
    for requirement in requirements:
        if candidate_items:
            similarities = [(phrase_similarity(requirement, item), item) for item in candidate_items]
            best_similarity, best_item = max(similarities, key=lambda x: x[0])
        else:
            best_similarity, best_item = 0.0, None
        best_pairs.append((requirement, best_item, best_similarity))

    matched = [req for req, item, _ in best_pairs if item and phrase_is_explicit_match(req, item)]
    missing = [req for req, item, _ in best_pairs if not item or not phrase_is_explicit_match(req, item)]
    score = 100.0 * sum(sim for _, _, sim in best_pairs) / len(best_pairs)
    return {"score": round(score, 4), "matched": matched, "missing": missing}


def safe_float(value):
    values = cell_values(value)
    if not values:
        return None
    try:
        number = float(values[0])
        return number if np.isfinite(number) else None
    except (TypeError, ValueError):
        return None

---
## 7. Structured field scoring

Experience and education use the same enum order defined in Phases 1/2. Meeting or exceeding a stated minimum receives full credit; being below it reduces the score gradually. Missing requirements return `None`, allowing the final aggregator to remove that component instead of assigning a false zero.

In [ ]:
def ordinal_score(candidate_value, job_value, order: dict):
    candidate_items = [str(v).strip().lower() for v in cell_values(candidate_value)]
    job_items = [str(v).strip().lower() for v in cell_values(job_value)]
    candidate_ranks = [order[v] for v in candidate_items if v in order]
    job_ranks = [order[v] for v in job_items if v in order]
    if not candidate_ranks or not job_ranks:
        return None

    candidate_rank = max(candidate_ranks)
    # Multiple job education values mean alternatives; use the lowest accepted requirement.
    required_rank = min(job_ranks)
    shortfall = max(required_rank - candidate_rank, 0)
    # Min-max normalized ordinal distance. With five ordered levels this
    # yields 100/75/50/25/0 without hand-selecting step penalties.
    max_distance = max(len(order) - 1, 1)
    return round(100.0 * (1.0 - shortfall / max_distance), 4)


def years_experience_score(candidate_years, job_minimum):
    candidate_years = safe_float(candidate_years)
    job_minimum = safe_float(job_minimum)
    if candidate_years is None or job_minimum is None:
        return None
    if job_minimum <= 0 or candidate_years >= job_minimum:
        return 100.0
    return round(100.0 * max(candidate_years, 0.0) / job_minimum, 4)


def education_score(candidate: dict, job_row: pd.Series) -> dict:
    level_score = ordinal_score(
        candidate.get("education_level"), job_row.get("education_level"), _edu_order
    )
    field_result = phrase_list_coverage(
        job_row.get("education_field"), candidate.get("education_field")
    )
    return {
        "level_score": level_score,
        "field_score": field_result["score"],
        "matched_fields": field_result["matched"],
        "missing_fields": field_result["missing"],
    }


def bool_value(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass
    return str(value).strip().lower() in {"true", "1", "yes"}


def experience_fit_score(candidate: dict, job_row: pd.Series) -> dict:
    years_score = years_experience_score(
        candidate.get("years_experience_total"), job_row.get("years_experience_min")
    )
    level_score = ordinal_score(
        candidate.get("experience_level"), job_row.get("experience_level"), _exp_order
    )
    # Explicit numeric requirements carry more information than an ordinal
    # category, so use years when both sides provide them; otherwise use level.
    score = years_score if years_score is not None else level_score
    return {"score": score, "years_score": years_score, "level_score": level_score}


def equal_weight_absolute_score(component_scores: dict) -> dict:
    active = {k: float(component_scores[k]) for k in RANKING_CRITERIA
              if component_scores.get(k) is not None and np.isfinite(float(component_scores[k]))}
    score = float(np.mean(list(active.values()))) if active else 0.0
    equal_weights = {k: (1.0 / len(active) if k in active else 0.0) for k in RANKING_CRITERIA}
    return {
        "match_score": round(score, 4),
        "scoring_coverage": round(len(active) / len(RANKING_CRITERIA), 4),
        "equal_weights": equal_weights,
    }


def critic_objective_weights(score_frame: pd.DataFrame) -> dict:
    """CRITIC: weight = contrast (std) × total conflict (1-correlation)."""
    available = [c for c in RANKING_CRITERIA if c in score_frame and score_frame[c].notna().any()]
    if not available:
        return {c: 0.0 for c in RANKING_CRITERIA}

    matrix = score_frame[available].astype(float) / 100.0
    matrix = matrix.apply(lambda col: col.fillna(col.mean()), axis=0)
    std = matrix.std(ddof=0)
    informative = std[std > 0].index.tolist()

    if len(matrix) < 2 or not informative:
        neutral = 1.0 / len(available)
        return {c: (neutral if c in available else 0.0) for c in RANKING_CRITERIA}

    corr = matrix[informative].corr().fillna(0.0)
    information = std[informative] * (1.0 - corr).sum(axis=1)
    if information.sum() <= 0:
        neutral = 1.0 / len(available)
        return {c: (neutral if c in available else 0.0) for c in RANKING_CRITERIA}

    raw = (information / information.sum()).to_dict()
    return {c: float(raw.get(c, 0.0)) for c in RANKING_CRITERIA}


def critic_score(row: pd.Series, weights: dict) -> float:
    active = [(float(row[c]), weights[c]) for c in RANKING_CRITERIA
              if c in row and pd.notna(row[c]) and weights.get(c, 0.0) > 0]
    if not active:
        return float(row.get("match_score", 0.0))
    total_weight = sum(weight for _, weight in active)
    return round(sum(score * weight for score, weight in active) / total_weight, 4)

---
## 8. Complete job-to-candidate matcher

This function produces both the component scores used for ranking and the evidence required downstream. Required/preferred skills are compared against the candidate's full normalized skill universe, while each skill bucket is also compared with its matching bucket to retain field-by-field transparency.

In [ ]:
CANDIDATE_CORE_SKILLS = sorted(set().union(
    normalized_skill_set(candidate_profile.get("hard_skills_norm")),
    normalized_skill_set(candidate_profile.get("soft_skills_norm")),
    normalized_skill_set(candidate_profile.get("tools_technologies_norm")),
))
CANDIDATE_ALL_SKILLS = sorted(set(CANDIDATE_CORE_SKILLS) | 
                              normalized_skill_set(candidate_profile.get("languages_norm")))


def match_job(job_row: pd.Series, candidate: dict) -> dict:
    job_languages = normalized_skill_set(job_row.get("languages_norm"))
    stated_required = normalized_skill_set(job_row.get("required_skills_norm"))
    core_requirements = stated_required - job_languages
    if not core_requirements:
        core_requirements = set().union(
            normalized_skill_set(job_row.get("hard_skills_norm")),
            normalized_skill_set(job_row.get("soft_skills_norm")),
            normalized_skill_set(job_row.get("tools_technologies_norm")),
        ) - job_languages

    core = skill_coverage(core_requirements, CANDIDATE_CORE_SKILLS)
    required = skill_coverage(job_row.get("required_skills_norm"), CANDIDATE_ALL_SKILLS)
    preferred = skill_coverage(job_row.get("preferred_skills_norm"), CANDIDATE_ALL_SKILLS)
    hard = skill_coverage(job_row.get("hard_skills_norm"), candidate.get("hard_skills_norm"))
    tools = skill_coverage(job_row.get("tools_technologies_norm"), candidate.get("tools_technologies_norm"))
    soft = skill_coverage(job_row.get("soft_skills_norm"), candidate.get("soft_skills_norm"))
    languages = skill_coverage(job_row.get("languages_norm"), candidate.get("languages_norm"))

    experience = experience_fit_score(candidate, job_row)
    education = education_score(candidate, job_row)
    job_professional_qualifications = [
        value for value in cell_values(job_row.get("qualifications"))
        if not is_education_credential(value)
    ]
    candidate_professional_qualifications = [
        value for value in cell_values(candidate.get("qualifications"))
        if not is_education_credential(value)
    ]
    qualifications = phrase_list_coverage(
        job_professional_qualifications, candidate_professional_qualifications
    )

    component_scores = {
        "core_skills_score": core["score"],
        "required_skills_score": required["score"],
        "preferred_skills_score": preferred["score"],
        "hard_skills_score": hard["score"],
        "tools_score": tools["score"],
        "soft_skills_score": soft["score"],
        "languages_score": languages["score"],
        "experience_fit_score": experience["score"],
        "experience_level_score": experience["level_score"],
        "years_experience_score": experience["years_score"],
        "education_level_score": education["level_score"],
        "education_field_score": education["field_score"],
        "education_score": education["field_score"],
        "qualifications_score": qualifications["score"],
    }

    absolute = equal_weight_absolute_score(component_scores)
    candidate_confidence = safe_float(candidate.get("extraction_confidence"))
    job_confidence = safe_float(job_row.get("extraction_confidence"))
    source_confidences = [v for v in (candidate_confidence, job_confidence) if v is not None]
    source_confidence = sum(source_confidences) / len(source_confidences) if source_confidences else 1.0
    experience_inferred = (
        bool_value(job_row.get("experience_level_inferred"))
        or bool_value(candidate.get("experience_level_inferred"))
    )
    active_reliabilities = []
    for criterion in RANKING_CRITERIA:
        if component_scores.get(criterion) is not None:
            active_reliabilities.append(
                source_confidence if criterion == "experience_fit_score" and experience_inferred else 1.0
            )
    evidence_reliability = float(np.mean(active_reliabilities)) if active_reliabilities else 0.0
    scoring_confidence = absolute["scoring_coverage"] * source_confidence * evidence_reliability

    evidence = {
        "matched_core_skills": core["matched"],
        "missing_core_skills": core["missing"],
        "matched_required_skills": required["matched"],
        "missing_required_skills": required["missing"],
        "matched_preferred_skills": preferred["matched"],
        "missing_preferred_skills": preferred["missing"],
        "matched_hard_skills": hard["matched"],
        "missing_hard_skills": hard["missing"],
        "matched_tools": tools["matched"],
        "missing_tools": tools["missing"],
        "matched_soft_skills": soft["matched"],
        "missing_soft_skills": soft["missing"],
        "matched_languages": languages["matched"],
        "missing_languages": languages["missing"],
        "matched_qualifications": qualifications["matched"],
        "missing_qualifications": qualifications["missing"],
        "matched_education_fields": education["matched_fields"],
        "missing_education_fields": education["missing_fields"],
    }

    return {
        **component_scores,
        "skill_match_score": core["score"] if core["score"] is not None else 0.0,
        "match_score": absolute["match_score"],
        "scoring_coverage": absolute["scoring_coverage"],
        "scoring_confidence": round(float(np.clip(scoring_confidence, 0.0, 1.0)), 4),
        **evidence,
        "match_evidence_json": json.dumps(evidence, ensure_ascii=False, sort_keys=True),
        "component_scores_json": json.dumps(component_scores, ensure_ascii=False, sort_keys=True),
        "effective_weights_json": json.dumps(absolute["equal_weights"], sort_keys=True),
    }

### 8.1 Sanity checks before scoring real jobs

In [ ]:
_perfect = skill_coverage(["python", "sql"], ["sql", "python", "excel"])
_partial = skill_coverage(["python", "sql"], ["python"])
_unknown = skill_coverage([], ["python"])
assert _perfect["score"] == 100.0 and not _perfect["missing"]
assert 0.0 < _partial["score"] < 100.0 and _partial["missing"] == ["sql"]
assert _unknown["score"] is None
assert ordinal_score("Senior Level", "Mid Level", _exp_order) == 100.0
assert ordinal_score("Entry Level", "Mid Level", _exp_order) == 75.0
assert years_experience_score(3, 3) == 100.0
assert years_experience_score(None, 3) is None
assert math.isclose(phrase_similarity("PMP", "PMP Certification"), 2 / 3)
assert phrase_is_explicit_match("PMP", "PMP Certification")
_critic_test = pd.DataFrame({
    "core_skills_score": [10.0, 50.0, 90.0],
    "languages_score": [100.0, 100.0, 100.0],
    "experience_fit_score": [90.0, 50.0, 10.0],
    "education_field_score": [20.0, 40.0, 80.0],
    "qualifications_score": [30.0, 30.0, 70.0],
})
_critic_weights = critic_objective_weights(_critic_test)
assert math.isclose(sum(_critic_weights.values()), 1.0, abs_tol=1e-9)
assert _critic_weights["languages_score"] == 0.0  # constant criterion adds no ranking information
print("Scoring sanity checks passed.")

---
## 9. Score and rank every surviving job

`match_score` is the parameter-free equal mean of the active, non-redundant criteria and remains interpretable as absolute evidence coverage. Ranking uses `objective_rank_score`, whose CRITIC weights are calculated from the surviving-job matrix. Preferred-skill coverage, confidence, Phase 3 similarity and `post_filter_rank` are deterministic tie-breakers only. Semantic similarity does **not** enter either Phase 5 score; it remains an independent Phase 6 signal.

In [ ]:
match_records = [match_job(row, candidate_profile) for _, row in filtered_df.iterrows()]
if match_records:
    score_df = pd.DataFrame(match_records, index=filtered_df.index)
else:
    # Preserve the complete Phase 5 schema even when Phase 4 has no survivors.
    component_cols = list(dict.fromkeys(RANKING_CRITERIA + [
        "required_skills_score", "preferred_skills_score", "hard_skills_score",
        "tools_score", "soft_skills_score", "experience_level_score",
        "years_experience_score", "education_level_score", "education_score",
    ]))
    evidence_cols = [
        "matched_core_skills", "missing_core_skills",
        "matched_required_skills", "missing_required_skills",
        "matched_preferred_skills", "missing_preferred_skills",
        "matched_hard_skills", "missing_hard_skills",
        "matched_tools", "missing_tools",
        "matched_soft_skills", "missing_soft_skills",
        "matched_languages", "missing_languages",
        "matched_qualifications", "missing_qualifications",
        "matched_education_fields", "missing_education_fields",
    ]
    extra_cols = [
        "skill_match_score", "match_score", "objective_rank_score",
        "scoring_coverage", "scoring_confidence", "match_evidence_json",
        "component_scores_json", "effective_weights_json",
    ]
    score_df = pd.DataFrame(index=filtered_df.index, columns=component_cols + evidence_cols + extra_cols)
matched_df = pd.concat([filtered_df.copy(), score_df], axis=1)
CRITIC_WEIGHTS = critic_objective_weights(score_df)
matched_df["objective_rank_score"] = matched_df.apply(
    lambda row: critic_score(row, CRITIC_WEIGHTS), axis=1
) if len(matched_df) else pd.Series(dtype=float)
matched_df["critic_weights_json"] = json.dumps(CRITIC_WEIGHTS, sort_keys=True)

if len(matched_df):
    matched_df = matched_df.sort_values(
        ["objective_rank_score", "match_score", "preferred_skills_score",
         "scoring_confidence", "similarity", "post_filter_rank"],
        ascending=[False, False, False, False, False, True],
        kind="mergesort",
    ).reset_index(drop=True)

matched_df.insert(0, "phase5_rank", range(1, len(matched_df) + 1))
matched_df.insert(1, "candidate_id", candidate_profile["candidate_id"])
matched_df["is_phase6_candidate"] = matched_df["phase5_rank"] <= CONFIG["phase6_pool_size"]

print(f"Scored {len(matched_df):,} jobs for candidate {candidate_profile['candidate_id']}")
if len(matched_df):
    print(f"Absolute match-score range: {matched_df['match_score'].min():.2f} – "
          f"{matched_df['match_score'].max():.2f}")
    print("CRITIC objective weights:", {k: round(v, 4) for k, v in CRITIC_WEIGHTS.items()})
    print(f"Jobs marked for Phase 6: {int(matched_df['is_phase6_candidate'].sum())}")

---
## 10. Review top matches

In [ ]:
display_cols = [
    "phase5_rank", "objective_rank_score", "match_score", "scoring_confidence", "similarity",
    "title_clean", "company_clean", "core_skills_score",
    "experience_fit_score", "education_field_score", "qualifications_score",
    "matched_core_skills", "missing_core_skills",
]
display_cols = [c for c in display_cols if c in matched_df.columns]
matched_df[display_cols].head(20).style.format({
    "objective_rank_score": "{:.2f}",
    "match_score": "{:.2f}",
    "scoring_confidence": "{:.2f}",
    "similarity": "{:.4f}",
    "core_skills_score": lambda x: "—" if pd.isna(x) else f"{x:.1f}",
    "experience_fit_score": lambda x: "—" if pd.isna(x) else f"{x:.1f}",
    "education_field_score": lambda x: "—" if pd.isna(x) else f"{x:.1f}",
    "qualifications_score": lambda x: "—" if pd.isna(x) else f"{x:.1f}",
})

### 10.1 Diagnostics

In [ ]:
if len(matched_df) == 0:
    print("[WARNING] Phase 4 produced no surviving jobs, so there is nothing to match.")
else:
    print(matched_df["match_score"].describe().round(2).to_string())
    no_required = int(matched_df["required_skills_score"].isna().sum())
    print("\nScoring confidence:")
    print(matched_df["scoring_confidence"].describe().round(3).to_string())
    print(f"Jobs with no explicit required_skills: {no_required}/{len(matched_df)}")

    if len(matched_df) < 2 or matched_df["objective_rank_score"].nunique() == 1:
        print("[NOTE] The current pool contains no measurable ranking contrast. "
              "CRITIC therefore uses its documented neutral equal-weight fallback.")
    else:
        print("Objective score distribution contains usable ranking contrast for Phase 6.")

---
## 11. Save outputs

In [ ]:
LIVE_OUT = OUT / "live_candidate"
LIVE_OUT.mkdir(parents=True, exist_ok=True)

matched_path = LIVE_OUT / "matched_jobs.parquet"
matched_df.to_parquet(matched_path, index=False, compression="snappy")

summary_cols = [
    "phase5_rank", "candidate_id", "job_id", "title_clean", "company_clean",
    "objective_rank_score", "match_score", "scoring_confidence", "similarity",
    "core_skills_score", "preferred_skills_score", "experience_fit_score",
    "education_field_score", "qualifications_score",
    "matched_core_skills", "missing_core_skills",
    "is_phase6_candidate",
]
summary_cols = [c for c in summary_cols if c in matched_df.columns]
summary_df = matched_df[summary_cols].copy()
for col in ["matched_core_skills", "missing_core_skills"]:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].map(lambda values: " | ".join(cell_values(values)))
summary_path = LIVE_OUT / "matching_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

RUN_METADATA = {
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "candidate_id": candidate_profile["candidate_id"],
    "n_input_jobs": int(len(filtered_df)),
    "n_matched_jobs": int(len(matched_df)),
    "n_phase6_candidates": int(matched_df["is_phase6_candidate"].sum()),
    "scoring_version": "phase5-evidence-based-v2",
    "ranking_criteria": RANKING_CRITERIA,
    "critic_weights": CRITIC_WEIGHTS,
    "config": CONFIG,
}
(OUT / "run_metadata.json").write_text(
    json.dumps(RUN_METADATA, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"Saved {matched_path}  ({len(matched_df)} rows)")
print(f"Saved {summary_path}  ({len(summary_df)} rows)")
print(f"Saved {OUT / 'run_metadata.json'}")

---
## 12. Data dictionary

In [ ]:
DATA_DICTIONARY = '''# Phase 5 outputs — Data Dictionary

## live_candidate/matched_jobs.parquet

One row per job that survived Phase 4, ordered by deterministic field-by-field
candidate match. All columns from `filtered_jobs.parquet` are preserved.

### Ranking and hand-off
| Column | Notes |
|---|---|
| phase5_rank | 1 = highest Phase 5 match. Contiguous, no gaps. |
| candidate_id | Candidate foreign key copied from Phase 2. |
| match_score | Equal-weight mean of active non-redundant evidence criteria in [0,100]; an absolute, parameter-free baseline. |
| objective_rank_score | CRITIC-weighted score in [0,100], used for Phase 5 ordering. Weights are learned from contrast/correlation in this candidate's surviving pool. |
| scoring_coverage | Fraction of the five ranking criteria supported by stated job fields. |
| scoring_confidence | Coverage × evidence reliability × source extraction confidence, in [0, 1]. |
| is_phase6_candidate | True for the leading pool that Phase 6 should re-rank. |
| similarity | Phase 3 cosine similarity, carried unchanged as an independent Phase 6 signal. |

### Component scores
The five independent ranking criteria are `core_skills_score`,
`languages_score`, `experience_fit_score`, `education_field_score`, and
`qualifications_score`. Diagnostic fields preserve required/preferred and
bucket-level skill coverage, ordinal/numeric experience, and education-level
eligibility. Education level does not enter Phase 5 scoring because Phase 4
already enforced it; degree phrases are likewise removed from professional
qualifications to prevent double-counting. Null criteria are unassessed rather than zero.

### Evidence for Phase 7
The `matched_*` and `missing_*` list columns are factual evidence for later
LLM explanations. `match_evidence_json` is the same evidence in a portable
JSON string. `component_scores_json`, `effective_weights_json`, and
`critic_weights_json` preserve both the equal and objective weighting paths.

## live_candidate/matching_summary.csv

Human-readable subset for quick inspection. List fields are pipe-delimited.
The Parquet file remains the authoritative machine-readable output.

## run_metadata.json

Run identifier, candidate id, row counts, scoring version and complete CONFIG.

## Phase 6 contract
- Read `matched_jobs.parquet`; do not recompute Phase 5 components.
- Re-rank rows where `is_phase6_candidate == True` (default: top 50).
- Keep `objective_rank_score`, `match_score`, `similarity`, `scoring_confidence`, and evidence columns.
- A cross-encoder/model score may be fused with these independent signals.
- Phase 6 outputs the final top 10-20 jobs for Phase 7 explanation.

## Matching contract
- No embeddings or LLM calls are made in Phase 5.
- Skill matching is weighted recall using Phase 1/2 `*_norm` fields and Phase 1 IDF.
- Missing job requirements are unassessed, not scored as zero.
- Experience uses the explicit years ratio when available; otherwise normalized ordinal distance.
- Education has no 70/30 split: level is diagnostic, field relevance is a separate criterion.
- No subjective fixed component weights or phrase thresholds are used.
- Phase 3 `similarity` is a tie-breaker only and is not part of either Phase 5 score.
'''

(OUT / "PHASE5_DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")
print("Data dictionary written.")

---
## 13. Final verification

In [ ]:
reloaded = pd.read_parquet(OUT / "live_candidate" / "matched_jobs.parquet")

assert len(reloaded) == len(filtered_df), "Phase 5 must score every Phase 4 survivor"
assert reloaded["job_id"].is_unique, "duplicate job_id in matched output"
assert reloaded["candidate_id"].nunique() <= 1, "single-candidate output contains multiple candidate ids"
assert reloaded["match_score"].between(0, 100).all(), "match_score outside [0, 100]"
assert reloaded["objective_rank_score"].between(0, 100).all(), "objective_rank_score outside [0, 100]"
assert reloaded["scoring_confidence"].between(0, 1).all(), "scoring_confidence outside [0, 1]"

if len(reloaded):
    assert reloaded["phase5_rank"].tolist() == list(range(1, len(reloaded) + 1))
    assert reloaded["objective_rank_score"].is_monotonic_decreasing
    assert int(reloaded["is_phase6_candidate"].sum()) == min(CONFIG["phase6_pool_size"], len(reloaded))
    print(f"rows       : {len(reloaded)}")
    print(f"top match  : {reloaded.iloc[0].get('title_clean')} "
          f"(rank_score={reloaded.iloc[0]['objective_rank_score']:.2f}, "
          f"absolute_match={reloaded.iloc[0]['match_score']:.2f}, "
          f"confidence={reloaded.iloc[0]['scoring_confidence']:.2f})")
else:
    print("rows       : 0 (Phase 4 had no survivors)")

print("STATUS: ready for Phase 6")

---
## 14. Persist outputs

Direct download is the simplest and most reliable option, matching the previous notebooks. The ZIP contains the live-candidate files, metadata and data dictionary.

In [ ]:
zip_path = shutil.make_archive("phase5_outputs", "zip", OUT)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size/1e6:.2f} MB)")

_colab_files.download(zip_path)

---
## Next step — PHASE 6

`matched_jobs.parquet` contains every Phase 4 survivor with a transparent score breakdown and evidence. Phase 6 should take the rows marked `is_phase6_candidate`, score each candidate/job pair with a lightweight cross-encoder (or another configured re-ranker), and retain `objective_rank_score`, absolute `match_score`, and the independent Phase 3 `similarity` as separate features rather than inventing another fixed weighted sum. It then emits the final top 10–20 jobs for Phase 7.

### Method references

- Robertson & Spärck Jones (1976), probabilistic relevance weighting / term specificity: https://doi.org/10.1002/asi.4630270302
- Diakoulaki, Mavrotas & Papayannakis (1995), CRITIC objective weighting: https://doi.org/10.1016/0305-0548(94)00059-H
- O*NET Content Model, separation of skills, education, experience and credentials: https://www.onetcenter.org/content.html
- Valizadegan et al. (2009), learning-to-rank evaluation/optimization with NDCG: https://proceedings.neurips.cc/paper/2009/hash/b3967a0e938dc2a6340e258630febd5a-Abstract.html